# Loading Dataset

In [16]:
from google.colab import drive

drive.mount('/content/drive')

print("Google Drive mounted successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully.


In [17]:
import os
import numpy as np

data_path = (
    "/content/drive/MyDrive/"
    "IITG dataset/cleaned_dataset/"
    "processed_dataset_v1/sequences_v1.npz"
)

print("Dataset exists:", os.path.exists(data_path))

data = np.load(data_path)

print("\nKeys inside sequences_v1.npz:")
print(data.files)

Dataset exists: True

Keys inside sequences_v1.npz:
['X', 'SoH', 'RUL', 'battery_id', 'split']


In [19]:
# ============================================================
# DAY 19 - CELL 3
# Create Train / Validation / Test Sets
# ============================================================

X = data["X"]
y = data["SoH"]
split = data["split"]

print("X shape:", X.shape)
print("SoH shape:", y.shape)
print("Split shape:", split.shape)

print("\nUnique split values:")
print(np.unique(split, return_counts=True))

X shape: (2287, 10, 2)
SoH shape: (2287,)
Split shape: (2287,)

Unique split values:
(array(['test', 'train', 'validation'], dtype='<U10'), array([ 392, 1516,  379]))


# Tunable LSTM Model

In [22]:
import torch.nn as nn

class TunableLSTM(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        num_layers,
        dropout,
        output_dim=1
    ):
        super(TunableLSTM, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.fc1 = nn.Linear(
            hidden_dim,
            32
        )

        self.relu = nn.ReLU()

        self.drop = nn.Dropout(dropout)

        self.fc2 = nn.Linear(
            32,
            output_dim
        )

    def forward(self, x):

        out, _ = self.lstm(x)

        out = out[:, -1, :]

        out = self.fc1(out)

        out = self.relu(out)

        out = self.drop(out)

        out = self.fc2(out)

        return out.squeeze()

# Hyperparameter Search Space

In [23]:
import itertools

search_space = {

    "hidden_dim": [32, 64],

    "num_layers": [1, 2],

    "dropout": [0.1, 0.3],

    "lr": [0.001, 0.005]
}

keys, values = zip(
    *search_space.items()
)

experiment_configs = [
    dict(zip(keys, v))
    for v in itertools.product(*values)
]

print("Total experiments:", len(experiment_configs))

print("\nConfigurations:")

for i, config in enumerate(
    experiment_configs,
    start=1
):
    print(f"{i}: {config}")

Total experiments: 16

Configurations:
1: {'hidden_dim': 32, 'num_layers': 1, 'dropout': 0.1, 'lr': 0.001}
2: {'hidden_dim': 32, 'num_layers': 1, 'dropout': 0.1, 'lr': 0.005}
3: {'hidden_dim': 32, 'num_layers': 1, 'dropout': 0.3, 'lr': 0.001}
4: {'hidden_dim': 32, 'num_layers': 1, 'dropout': 0.3, 'lr': 0.005}
5: {'hidden_dim': 32, 'num_layers': 2, 'dropout': 0.1, 'lr': 0.001}
6: {'hidden_dim': 32, 'num_layers': 2, 'dropout': 0.1, 'lr': 0.005}
7: {'hidden_dim': 32, 'num_layers': 2, 'dropout': 0.3, 'lr': 0.001}
8: {'hidden_dim': 32, 'num_layers': 2, 'dropout': 0.3, 'lr': 0.005}
9: {'hidden_dim': 64, 'num_layers': 1, 'dropout': 0.1, 'lr': 0.001}
10: {'hidden_dim': 64, 'num_layers': 1, 'dropout': 0.1, 'lr': 0.005}
11: {'hidden_dim': 64, 'num_layers': 1, 'dropout': 0.3, 'lr': 0.001}
12: {'hidden_dim': 64, 'num_layers': 1, 'dropout': 0.3, 'lr': 0.005}
13: {'hidden_dim': 64, 'num_layers': 2, 'dropout': 0.1, 'lr': 0.001}
14: {'hidden_dim': 64, 'num_layers': 2, 'dropout': 0.1, 'lr': 0.005}
15: 

# Grid Search

In [24]:
import time
import copy

# Device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Training settings
EPOCHS = 20

experiment_logs = []

best_val_loss = float("inf")
best_config = None
best_model_state = None

input_dim = X_train.shape[2]

print("Input dimension:", input_dim)
print("Device:", device)
print("Epochs:", EPOCHS)
print("Total experiments:", len(experiment_configs))


for idx, config in enumerate(experiment_configs):

    print("\n" + "=" * 60)
    print(f"Experiment {idx + 1}/{len(experiment_configs)}")
    print("=" * 60)

    print("Configuration:", config)

    # --------------------------------------------------------
    # Create LSTM
    # --------------------------------------------------------

    model = TunableLSTM(
        input_dim=input_dim,
        hidden_dim=config["hidden_dim"],
        num_layers=config["num_layers"],
        dropout=config["dropout"]
    ).to(device)

    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["lr"]
    )

    start_time = time.time()

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    for epoch in range(EPOCHS):

        model.train()

        for batch_X, batch_y in train_loader:

            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()

            outputs = model(batch_X)

            loss = criterion(
                outputs,
                batch_y
            )

            loss.backward()

            optimizer.step()

    training_time = time.time() - start_time

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    model.eval()

    total_val_loss = 0.0

    with torch.no_grad():

        for batch_X, batch_y in val_loader:

            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_X)

            loss = criterion(
                outputs,
                batch_y
            )

            total_val_loss += (
                loss.item() * batch_X.size(0)
            )

    val_loss = (
        total_val_loss /
        len(val_loader.dataset)
    )

    # --------------------------------------------------------
    # Count parameters
    # --------------------------------------------------------

    num_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    # --------------------------------------------------------
    # Save experiment result
    # --------------------------------------------------------

    result = config.copy()

    result["Experiment_ID"] = idx + 1
    result["Validation_MSE"] = val_loss
    result["Training_Time_s"] = training_time
    result["Parameters"] = num_params

    experiment_logs.append(result)

    print(f"Validation MSE : {val_loss:.6f}")
    print(f"Training Time  : {training_time:.2f} s")
    print(f"Parameters     : {num_params:,}")

    # --------------------------------------------------------
    # Check whether this is the best model
    # --------------------------------------------------------

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_config = config.copy()

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        print(">>> NEW BEST MODEL")

Input dimension: 2
Device: cpu
Epochs: 20
Total experiments: 16

Experiment 1/16
Configuration: {'hidden_dim': 32, 'num_layers': 1, 'dropout': 0.1, 'lr': 0.001}
Validation MSE : 621.147420
Training Time  : 3.61 s
Parameters     : 5,697
>>> NEW BEST MODEL

Experiment 2/16
Configuration: {'hidden_dim': 32, 'num_layers': 1, 'dropout': 0.1, 'lr': 0.005}
Validation MSE : 619.522188
Training Time  : 5.99 s
Parameters     : 5,697
>>> NEW BEST MODEL

Experiment 3/16
Configuration: {'hidden_dim': 32, 'num_layers': 1, 'dropout': 0.3, 'lr': 0.001}
Validation MSE : 598.999644
Training Time  : 3.17 s
Parameters     : 5,697
>>> NEW BEST MODEL

Experiment 4/16
Configuration: {'hidden_dim': 32, 'num_layers': 1, 'dropout': 0.3, 'lr': 0.005}
Validation MSE : 793.643387
Training Time  : 3.17 s
Parameters     : 5,697

Experiment 5/16
Configuration: {'hidden_dim': 32, 'num_layers': 2, 'dropout': 0.1, 'lr': 0.001}
Validation MSE : 611.524169
Training Time  : 6.04 s
Parameters     : 14,145

Experiment 6/16
C

In [25]:
# ============================================================
# DAY 19 - CELL 9
# Experiment Results
# ============================================================

results_df = pd.DataFrame(experiment_logs)

results_df = results_df[
    [
        "Experiment_ID",
        "hidden_dim",
        "num_layers",
        "dropout",
        "lr",
        "Validation_MSE",
        "Training_Time_s",
        "Parameters"
    ]
]

results_df = results_df.sort_values(
    by="Validation_MSE"
).reset_index(drop=True)

print("DAY 19 - LSTM HYPERPARAMETER TUNING RESULTS")
print("=" * 70)

display(results_df)

DAY 19 - LSTM HYPERPARAMETER TUNING RESULTS


,Experiment_ID,hidden_dim,num_layers,dropout,lr,Validation_MSE,Training_Time_s,Parameters
0,6,32,2,0.1,0.005,585.195812,6.067727,14145
1,16,64,2,0.3,0.005,588.898663,10.167887,52801
2,11,64,1,0.3,0.001,594.892803,5.689394,19521
3,3,32,1,0.3,0.001,598.999644,3.172081,5697
4,15,64,2,0.3,0.001,600.257689,9.365660,52801
5,8,32,2,0.3,0.005,601.518248,6.821894,14145
6,7,32,2,0.3,0.001,604.819521,5.422822,14145
7,5,32,2,0.1,0.001,611.524169,6.038989,14145
8,13,64,2,0.1,0.001,617.798124,10.142644,52801
9,12,64,1,0.3,0.005,619.289055,4.659851,19521


In [26]:
best_experiment = results_df.iloc[0]

print("BEST LSTM CONFIGURATION")
print("=" * 50)

print(
    "Experiment ID :",
    int(best_experiment["Experiment_ID"])
)

print(
    "Hidden Size   :",
    int(best_experiment["hidden_dim"])
)

print(
    "Num Layers    :",
    int(best_experiment["num_layers"])
)

print(
    "Dropout       :",
    best_experiment["dropout"]
)

print(
    "Learning Rate :",
    best_experiment["lr"]
)

print(
    "Validation MSE:",
    best_experiment["Validation_MSE"]
)

print(
    "Training Time :",
    best_experiment["Training_Time_s"]
)

print(
    "Parameters    :",
    int(best_experiment["Parameters"])
)

BEST LSTM CONFIGURATION
Experiment ID : 6
Hidden Size   : 32
Num Layers    : 2
Dropout       : 0.1
Learning Rate : 0.005
Validation MSE: 585.1958123763193
Training Time : 6.0677268505096436
Parameters    : 14145


# Saving the Result

In [28]:
# ============================================================
# DAY 19 - CELL 12
# Save Experiment Results
# ============================================================

results_path = (
    "/content/drive/MyDrive/"
    "IITG dataset/cleaned_dataset/"
    "processed_dataset_v1/"
    "day19_lstm_hyperparameter_results.csv"
)

results_df.to_csv(
    results_path,
    index=False
)

print("Experiment log saved successfully.")
print(results_path)

Experiment log saved successfully.
/content/drive/MyDrive/IITG dataset/cleaned_dataset/processed_dataset_v1/day19_lstm_hyperparameter_results.csv
